<a href="https://colab.research.google.com/github/minyi-k03/LargeLanguageModel/blob/Project-Based-Learning(PBL)/OpenAI_Responses_API_File_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenAI Responses API - File Search 예제

## 작성자 : AISchool ( http://aischool.ai/%ec%98%a8%eb%9d%bc%ec%9d%b8-%ea%b0%95%ec%9d%98-%ec%b9%b4%ed%85%8c%ea%b3%a0%eb%a6%ac/ )

## File Search API Reference : https://platform.openai.com/docs/guides/tools-file-search#metadata-filtering

## File Search API pricing : https://platform.openai.com/docs/pricing#built-in-tools

In [1]:
!pip install --upgrade openai

In [2]:
!pip show openai

Name: openai
Version: 2.15.0
Summary: The official Python library for the openai API
Home-page: https://github.com/openai/openai-python
Author: 
Author-email: OpenAI <support@openai.com>
License: Apache-2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: anyio, distro, httpx, jiter, pydantic, sniffio, tqdm, typing-extensions
Required-by: 


# Quick start

In [3]:
from openai import OpenAI

#API Key 설정
OPENAI_KEY = "Input Your Key"

#클라이언트 연결
client = OpenAI(api_key=OPENAI_KEY)

print("OpenAI 설정 완료!")

OpenAI 설정 완료!


## vector store 준비

테스트용 파일 : https://cdn.openai.com/API/docs/deep_research_blog.pdf

In [4]:
import requests
from io import BytesIO

def create_file(client, file_path):
    if file_path.startswith("http://") or file_path.startswith("https://"):
        # Download the file content from the URL
        response = requests.get(file_path)
        file_content = BytesIO(response.content)
        file_name = file_path.split("/")[-1]
        file_tuple = (file_name, file_content)
        result = client.files.create(
            file=file_tuple,
            purpose="assistants"
        )
    else:
        # Handle local file path
        with open(file_path, "rb") as file_content:
            result = client.files.create(
                file=file_content,
                purpose="assistants"
            )
    print(result.id)
    return result.id

# Replace with your own file path or URL
file_id = create_file(client, "https://cdn.openai.com/API/docs/deep_research_blog.pdf")

file-Q9YSJGthNHJTEy88crXLSh


In [5]:
vector_store = client.vector_stores.create(
    name="knowledge_base"
)
print(vector_store.id)

vs_697870442c688191a6aa7e16124be0d8


In [6]:
#해당 URL의 파일 아이디 Embedding해서 저
client.vector_stores.files.create(
    vector_store_id=vector_store.id,
    file_id=file_id
)

VectorStoreFile(id='file-Q9YSJGthNHJTEy88crXLSh', created_at=1769500759, last_error=None, object='vector_store.file', status='in_progress', usage_bytes=0, vector_store_id='vs_697870442c688191a6aa7e16124be0d8', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))

In [7]:
result = client.vector_stores.files.list(
    vector_store_id=vector_store.id
)
print(result)
print(result.data[0].status)

SyncCursorPage[VectorStoreFile](data=[VectorStoreFile(id='file-Q9YSJGthNHJTEy88crXLSh', created_at=1769500759, last_error=None, object='vector_store.file', status='completed', usage_bytes=90446, vector_store_id='vs_697870442c688191a6aa7e16124be0d8', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))], has_more=False, object='list', first_id='file-Q9YSJGthNHJTEy88crXLSh', last_id='file-Q9YSJGthNHJTEy88crXLSh')
completed


## file search tool 테스트하기

In [8]:
response = client.responses.create(
    model="gpt-4o-mini",
    input="OpenAI의 deep research가 뭐야?",
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store.id]
    }]
)
print(response)

Response(id='resp_091485a90b5704d0006978709ca36081a38951e35e08f6429d', created_at=1769500828.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseFileSearchToolCall(id='fs_091485a90b5704d0006978709dc1ec81a38e1c2cd4c6861672', queries=['OpenAI deep research', 'OpenAI의 deep research가 뭐야?', 'OpenAI deep research overview'], status='completed', type='file_search_call', results=None), ResponseOutputMessage(id='msg_091485a90b5704d000697870a176ec81a385d90cbd5e28b5ce', content=[ResponseOutputText(annotations=[AnnotationFileCitation(file_id='file-Q9YSJGthNHJTEy88crXLSh', filename='deep_research_blog.pdf', index=158, type='file_citation'), AnnotationFileCitation(file_id='file-Q9YSJGthNHJTEy88crXLSh', filename='deep_research_blog.pdf', index=250, type='file_citation'), AnnotationFileCitation(file_id='file-Q9YSJGthNHJTEy88crXLSh', filename='deep_research_blog.pdf', index=347, type='file_citation'), AnnotationFileC

In [9]:
print(response.output[1].content[0].text)

OpenAI의 "Deep Research"는 ChatGPT의 새로운 기능으로, 복잡한 작업을 위해 인터넷에서 다단계 연구를 수행할 수 있습니다. 이 기능은 사용자가 제시한 프롬프트에 따라 수백 개의 온라인 소스를 찾아 분석하고 종합하여 연구 분석가 수준의 포괄적인 보고서를 작성합니다.

주요 특징은 다음과 같습니다:

1. **자동화된 정보 검색**: Deep Research는 여러 출처에서 정보를 독립적으로 발견하고, 이를 분석하여 종합합니다.
  
2. **정밀한 분석**: 이 기능은 금융, 과학, 정책, 엔지니어링과 같은 지식 집약적인 분야에 최적화되어 있어, 까다로운 조사 작업을 신속하게 수행할 수 있습니다.

3. **문서화된 결과**: 생성되는 각 보고서는 명확한 출처 인용 및 사고의 요약이 포함되어 있어, 검토 및 참조가 용이합니다.

4. **시간 절약**: 복잡한 웹 연구를 단일 질의를 통해 신속하게 처리할 수 있어, 사용자에게 귀중한 시간을 절약해줍니다.

이와 같은 기능은 OpenAI가 AGI(인공지능 일반화) 개발을 위한 중요한 단계로 보고 있으며, 새로운 지식을 창출하는 데 기여할 것으로 기대하고 있습니다.


In [10]:
response

Response(id='resp_091485a90b5704d0006978709ca36081a38951e35e08f6429d', created_at=1769500828.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseFileSearchToolCall(id='fs_091485a90b5704d0006978709dc1ec81a38e1c2cd4c6861672', queries=['OpenAI deep research', 'OpenAI의 deep research가 뭐야?', 'OpenAI deep research overview'], status='completed', type='file_search_call', results=None), ResponseOutputMessage(id='msg_091485a90b5704d000697870a176ec81a385d90cbd5e28b5ce', content=[ResponseOutputText(annotations=[AnnotationFileCitation(file_id='file-Q9YSJGthNHJTEy88crXLSh', filename='deep_research_blog.pdf', index=158, type='file_citation'), AnnotationFileCitation(file_id='file-Q9YSJGthNHJTEy88crXLSh', filename='deep_research_blog.pdf', index=250, type='file_citation'), AnnotationFileCitation(file_id='file-Q9YSJGthNHJTEy88crXLSh', filename='deep_research_blog.pdf', index=347, type='file_citation'), AnnotationFileC

## return값 parsing을 위한 pretty_print_response 함수 정의

In [11]:
from datetime import datetime
import pprint

def pretty_print_response(response):
    print("\n📦 Response Summary")
    print("=" * 50)
    print(f"🆔 ID: {response.id}")
    print(f"📅 Created At: {datetime.fromtimestamp(response.created_at)}")
    print(f"🤖 Model: {response.model}")
    print(f"📂 Object Type: {response.object}")
    print()

    for idx, item in enumerate(response.output):
        print(f"\n🔍 Tool Call {idx + 1}")
        print("-" * 50)
        if hasattr(item, "queries"):
            print(f"🔎 Queries: {item.queries}")
            print(f"📊 Status: {item.status}")
            print(f"📁 Type: {item.type}")

            # ✅ Check if results exist and are non-empty
            if hasattr(item, "results") and item.results:
                for res_idx, result in enumerate(item.results):
                    print(f"\n📄 Result {res_idx + 1}:")
                    print(f"    🗂 File Name: {result.filename}")
                    print(f"    📎 File ID: {result.file_id}")
                    print(f"    📈 Score: {round(result.score, 3)}")
                    text = result.text.strip().replace("\n", " ") + "..."
                    print(f"    📃 Text: {text}")
            else:
                print("⚠️  No results found.")
        else:
            # For standard output text responses
            if hasattr(item, "content"):
                for content in item.content:
                    print(f"\n📝 Text Content:\n{content.text}...")

In [12]:
pretty_print_response(response)


📦 Response Summary
🆔 ID: resp_091485a90b5704d0006978709ca36081a38951e35e08f6429d
📅 Created At: 2026-01-27 08:00:28
🤖 Model: gpt-4o-mini-2024-07-18
📂 Object Type: response


🔍 Tool Call 1
--------------------------------------------------
🔎 Queries: ['OpenAI deep research', 'OpenAI의 deep research가 뭐야?', 'OpenAI deep research overview']
📊 Status: completed
📁 Type: file_search_call
⚠️  No results found.

🔍 Tool Call 2
--------------------------------------------------

📝 Text Content:
OpenAI의 "Deep Research"는 ChatGPT의 새로운 기능으로, 복잡한 작업을 위해 인터넷에서 다단계 연구를 수행할 수 있습니다. 이 기능은 사용자가 제시한 프롬프트에 따라 수백 개의 온라인 소스를 찾아 분석하고 종합하여 연구 분석가 수준의 포괄적인 보고서를 작성합니다.

주요 특징은 다음과 같습니다:

1. **자동화된 정보 검색**: Deep Research는 여러 출처에서 정보를 독립적으로 발견하고, 이를 분석하여 종합합니다.
  
2. **정밀한 분석**: 이 기능은 금융, 과학, 정책, 엔지니어링과 같은 지식 집약적인 분야에 최적화되어 있어, 까다로운 조사 작업을 신속하게 수행할 수 있습니다.

3. **문서화된 결과**: 생성되는 각 보고서는 명확한 출처 인용 및 사고의 요약이 포함되어 있어, 검토 및 참조가 용이합니다.

4. **시간 절약**: 복잡한 웹 연구를 단일 질의를 통해 신속하게 처리할 수 있어, 사용자에게 귀중한 시간을 절약해줍니다.

이와 같은 기능은 OpenAI

In [13]:
response = client.responses.create(
    model="gpt-4o-mini",
    input="OpenAI의 deep research가 뭐야?",
)
print(response)

Response(id='resp_078b544611a0d21e00697871a294208193bbe7fab961801e06', created_at=1769501090.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseOutputMessage(id='msg_078b544611a0d21e00697871a354f48193807e6d54285d322b', content=[ResponseOutputText(annotations=[], text='OpenAI의 Deep Research는 인공지능(AI) 기술의 한계를 확장하고, 새로운 알고리즘 및 모델을 개발하는 연구 분야를 의미합니다. 이 연구는 여러 가지 주제를 포함하며, 자율학습, 자연어 처리, 강화 학습, 컴퓨터 비전 등 다양한 영역에서 진행됩니다. \n\nDeep Research의 목표는 인공지능의 성능을 향상시키고, 이를 통해 인간 사회에 긍정적인 영향을 미칠 수 있는 혁신적인 기술을 개발하는 것입니다. OpenAI는 윤리적 고려사항도 중요하게 생각하며, AI 기술의 안전하고 유익한 사용을 위해 다양한 연구 결과를 공유하고 있습니다.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, completed_at=1769501094.0, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id=None, prompt=None, promp

In [16]:
print(response.output[0].content[0].text)
#GPT-4o-mini 모델에는 내장 지식에 없기 때문에 Deap Search에 대한 답변이 유연하지 않다

OpenAI의 Deep Research는 인공지능(AI) 기술의 한계를 확장하고, 새로운 알고리즘 및 모델을 개발하는 연구 분야를 의미합니다. 이 연구는 여러 가지 주제를 포함하며, 자율학습, 자연어 처리, 강화 학습, 컴퓨터 비전 등 다양한 영역에서 진행됩니다. 

Deep Research의 목표는 인공지능의 성능을 향상시키고, 이를 통해 인간 사회에 긍정적인 영향을 미칠 수 있는 혁신적인 기술을 개발하는 것입니다. OpenAI는 윤리적 고려사항도 중요하게 생각하며, AI 기술의 안전하고 유익한 사용을 위해 다양한 연구 결과를 공유하고 있습니다.


# include=["file_search_call.results"] 파라미터 추가하기

In [18]:
response = client.responses.create(
    model="gpt-4o-mini",
    input="OpenAI의 deep research가 뭐야?",
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store.id]
    }],
    include=["file_search_call.results"] #File Search Tool 사용 과정 트래킹
)
print(response)

Response(id='resp_0c440a7581ef9b0c00697872622638819dac88b28a37dc0b18', created_at=1769501282.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseFileSearchToolCall(id='fs_0c440a7581ef9b0c0069787263390c819d947c7c3b5306ce36', queries=['OpenAI의 deep research가 뭐야?', 'OpenAI deep research', 'OpenAI 연구'], status='completed', type='file_search_call', results=[Result(attributes={}, file_id='file-Q9YSJGthNHJTEy88crXLSh', filename='deep_research_blog.pdf', score=0.924, text="Today we’re launching deep research in ChatGPT, a new agentic capability that\r\nconducts multi-step research on the internet for complex tasks. It accomplishes in\r\ntens of minutes what would take a human many hours.\r\nDeep research is OpenAI's next agent that can do work for you independently—you\r\ngive it a prompt, and ChatGPT will find, analyze, and synthesize hundreds of online\r\nsources to create a comprehensive report at the lev

In [19]:
response

Response(id='resp_0c440a7581ef9b0c00697872622638819dac88b28a37dc0b18', created_at=1769501282.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseFileSearchToolCall(id='fs_0c440a7581ef9b0c0069787263390c819d947c7c3b5306ce36', queries=['OpenAI의 deep research가 뭐야?', 'OpenAI deep research', 'OpenAI 연구'], status='completed', type='file_search_call', results=[Result(attributes={}, file_id='file-Q9YSJGthNHJTEy88crXLSh', filename='deep_research_blog.pdf', score=0.924, text="Today we’re launching deep research in ChatGPT, a new agentic capability that\r\nconducts multi-step research on the internet for complex tasks. It accomplishes in\r\ntens of minutes what would take a human many hours.\r\nDeep research is OpenAI's next agent that can do work for you independently—you\r\ngive it a prompt, and ChatGPT will find, analyze, and synthesize hundreds of online\r\nsources to create a comprehensive report at the lev

In [20]:
#가져온 파일의 연관도를 점수로 나타내고 가장 높은 점수부터 출력한다
pretty_print_response(response)


📦 Response Summary
🆔 ID: resp_0c440a7581ef9b0c00697872622638819dac88b28a37dc0b18
📅 Created At: 2026-01-27 08:08:02
🤖 Model: gpt-4o-mini-2024-07-18
📂 Object Type: response


🔍 Tool Call 1
--------------------------------------------------
🔎 Queries: ['OpenAI의 deep research가 뭐야?', 'OpenAI deep research', 'OpenAI 연구']
📊 Status: completed
📁 Type: file_search_call

📄 Result 1:
    🗂 File Name: deep_research_blog.pdf
    📎 File ID: file-Q9YSJGthNHJTEy88crXLSh
    📈 Score: 0.924
 Compared to deep research, GPT‑4o is ideal for real-time, multimodal conversations....

📄 Result 2:
    🗂 File Name: deep_research_blog.pdf
    📎 File ID: file-Q9YSJGthNHJTEy88crXLSh
    📈 Score: 0.887
 and any uploaded files....

📄 Result 3:
    🗂 File Name: deep_research_blog.pdf
    📎 File ID: file-Q9YSJGthNHJTEy88crXLSh
    📈 Score: 0.843
 https://openai.com/index/introducing-deep-research/ 24/38...

📄 Result 4:
    🗂 File Name: deep_research_blog.pdf
    📎 File ID: file-Q9YSJGthNHJTEy88crXLSh
    📈 Score: 0.831

In [21]:
response = client.responses.create(
    model="gpt-4o-mini",
    input="google의 설립일을 알려줘",
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store.id]
    }],
    include=["file_search_call.results"]
)
print(response)

Response(id='resp_0a4078ffa6a9cc9000697872b19e9081a289715afd71b990cd', created_at=1769501361.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseFileSearchToolCall(id='fs_0a4078ffa6a9cc9000697872b272e481a2b4a906cdf3621a04', queries=['구글의 설립일', 'google founding date', 'google 설립일'], status='completed', type='file_search_call', results=[Result(attributes={}, file_id='file-Q9YSJGthNHJTEy88crXLSh', filename='deep_research_blog.pdf', score=0.1125, text="Today we’re launching deep research in ChatGPT, a new agentic capability that\r\nconducts multi-step research on the internet for complex tasks. It accomplishes in\r\ntens of minutes what would take a human many hours.\r\nDeep research is OpenAI's next agent that can do work for you independently—you\r\ngive it a prompt, and ChatGPT will find, analyze, and synthesize hundreds of online\r\nsources to create a comprehensive report at the level of a research 

In [22]:
#가져온 파일의 연관도를 점수로 나타내고 가장 높은 점수부터 출력한다
pretty_print_response(response)


📦 Response Summary
🆔 ID: resp_0a4078ffa6a9cc9000697872b19e9081a289715afd71b990cd
📅 Created At: 2026-01-27 08:09:21
🤖 Model: gpt-4o-mini-2024-07-18
📂 Object Type: response


🔍 Tool Call 1
--------------------------------------------------
🔎 Queries: ['구글의 설립일', 'google founding date', 'google 설립일']
📊 Status: completed
📁 Type: file_search_call

📄 Result 1:
    🗂 File Name: deep_research_blog.pdf
    📎 File ID: file-Q9YSJGthNHJTEy88crXLSh
    📈 Score: 0.113
 Compared to deep research, GPT‑4o is ideal for real-time, multimodal conversations....

📄 Result 2:
    🗂 File Name: deep_research_blog.pdf
    📎 File ID: file-Q9YSJGthNHJTEy88crXLSh
    📈 Score: 0.086
 simplified version....

📄 Result 3:
    🗂 File Name: deep_research_blog.pdf
    📎 File ID: file-Q9YSJGthNHJTEy88crXLSh
    📈 Score: 0.065
 chemistry, humanities and social sciences, and mathematics....

📄 Result 4:
    🗂 File Name: deep_research_blog.pdf
    📎 File ID: file-Q9YSJGthNHJTEy88crXLSh
    📈 Score: 0.047
 and mixed gas pred

In [23]:
response = client.responses.create(
    model="gpt-4o-mini",
    input="황제펭귄에 대해 알려줘", #불러오는 파일과 연관 없는 질문
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store.id]
    }],
    include=["file_search_call.results"]
)
print(response)

Response(id='resp_0b7596ba3283b2ca00697873051708819583b8923b40bf0e6b', created_at=1769501445.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseFileSearchToolCall(id='fs_0b7596ba3283b2ca0069787305fb6c819593666c367dcfb577', queries=['황제펭귄', '황제펭귄의 특징', '황제펭귄 서식지', '황제펭귄 생태'], status='completed', type='file_search_call', results=[Result(attributes={}, file_id='file-Q9YSJGthNHJTEy88crXLSh', filename='deep_research_blog.pdf', score=0.4927, text='This is attributed to “dual-mode” sorption: some\r\npenetrant dissolves in the dense polymer matrix (obeying Henry’s law)\r\nwhile additional penetrant saturates specific microvoid “sites” (a\r\nLangmuir-type adsorption in the polymer’s nonequilibrium free\r\nvolume). By contrast, when two or more gases are present,they\r\ncompete forthose limited Langmuir sorption sites.As a result, each\r\ncomponent’s sorbed concentration in a mixture is generally lower\r\ntha

In [24]:
#가져온 파일의 연관도를 점수로 나타내고 가장 높은 점수부터 출력한다
pretty_print_response(response)


📦 Response Summary
🆔 ID: resp_0b7596ba3283b2ca00697873051708819583b8923b40bf0e6b
📅 Created At: 2026-01-27 08:10:45
🤖 Model: gpt-4o-mini-2024-07-18
📂 Object Type: response


🔍 Tool Call 1
--------------------------------------------------
🔎 Queries: ['황제펭귄', '황제펭귄의 특징', '황제펭귄 서식지', '황제펭귄 생태']
📊 Status: completed
📁 Type: file_search_call

📄 Result 1:
    🗂 File Name: deep_research_blog.pdf
    📎 File ID: file-Q9YSJGthNHJTEy88crXLSh
    📈 Score: 0.493
 range....

📄 Result 2:
    🗂 File Name: deep_research_blog.pdf
    📎 File ID: file-Q9YSJGthNHJTEy88crXLSh
    📈 Score: 0.366
 Compared to deep research, GPT‑4o is ideal for real-time, multimodal conversations....

📄 Result 3:
    🗂 File Name: deep_research_blog.pdf
    📎 File ID: file-Q9YSJGthNHJTEy88crXLSh
    📈 Score: 0.351
 simplified version....

📄 Result 4:
    🗂 File Name: deep_research_blog.pdf
    📎 File ID: file-Q9YSJGthNHJTEy88crXLSh
    📈 Score: 0.227
 cannot capture....

📄 Result 5:
    🗂 File Name: deep_research_blog.pdf
    📎